# Warnings: mould detection from photos (first test on Kaggle)

**What this notebook does.** It trains a small model that draws a box around mould in a photo and says whether it is **Trichoderma** (green mould) or **mould** (everything else: Mold, Cobweb, Neurospora). It is the contamination part of the **Warnings** model for the cultivation planner.

**Data.** Three public Roboflow sets by the owner *baibua* (CC BY 4.0: credit the owner whenever results are shown). The newest version of each is used.

| Set | Roboflow project | Note |
|---|---|---|
| wwzez | baibua / mold-detection-training-wwzez | Trichoderma boxes look right |
| f3os8 | baibua / mold-detect-s-f3os8 | small set |
| 4n7zr | baibua / molds-4n7zr | real mould labels on many backgrounds (mycelium on many substrates and containers). This variety is wanted: user photos will look like that |

Plus **clean dish photos** (no mould, no boxes) from your own private Kaggle dataset `d005-clean-dishes`: the 8 substrate folders (FD, FLT, FLU, FSW, Flax, H, S, W) of d005 (Elsacker 2019), whole dishes photographed from above. They teach the model that a healthy white dish is not mould. d003 is left out: its images are thin curve strips (about 78×435 px), not whole dishes.

**Honest counting.** Roboflow adds flipped or rotated copies of one photo; we keep one copy per source photo. The same photo can also sit in two sets, or appear resized; a perceptual hash finds those and we keep one. The split into train and held-out is by photo, after duplicates are removed. Clean photos are split by dish, so photos of the same dish never sit on both sides.

**Never used here:** syldd photos. They are a final check only, and the notebook stops if a clean-photo path contains "syldd".

**Limit to remember.** The held-out 20% also picks the best epoch, so scores are slightly optimistic. The leave-one-source-out cell at the end (once a second licensed mould set exists) is the stronger test.

**Before you press Run All (once):**
1. Right panel, *Session options*: Accelerator = **GPU T4 x2** (or P100), Internet = **On**.
2. *Add-ons, Secrets*: switch `ROBOFLOW_API_KEY` on for this notebook. The key is never printed or saved.
3. *Add Input*: add your clean-dish dataset, then check `CLEAN_DIR` in the clearly marked cell (step 5): copy the real path with the copy icon in the Input panel.

At the end a small table with precision, recall, mAP50 and the false-alarm counts sums up everything; a screenshot of it is enough. The best weights are saved as `mould_best.pt` in the Output tab.

In [ ]:
# 1. Install the libraries we need (Kaggle has the rest)
!pip install -q roboflow ultralytics imagehash

In [ ]:
# 2. Settings: change these lines only if you want to
import json, random, re, shutil
from collections import Counter
from pathlib import Path

SETS = {
    # short name: (Roboflow workspace, project)
    "wwzez": ("baibua", "mold-detection-training-wwzez"),
    "f3os8": ("baibua", "mold-detect-s-f3os8"),
    "4n7zr": ("baibua", "molds-4n7zr"),
}
# their class name (lower case) -> our class number
CLASS_MAP = {"trichoderma": 0, "mold": 1, "mould": 1, "cobweb": 1, "neurospora": 1}
OUR_NAMES = {0: "Trichoderma", 1: "mould"}
HASH_DISTANCE = 4        # photos whose perceptual hashes differ in <= this many of 64 bits count as the same photo
HELD_OUT = 0.20          # share of photos (and of clean dishes) kept for testing
EPOCHS = 50
IMG_SIZE = 640
MODEL = "yolo11n.pt"     # smallest box model: fast, good enough for a first test
SEED = 0
WORK = Path("/kaggle/working")
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
random.seed(SEED)

## Step 1-2. Download the three sets
The key is read from the Kaggle secret and handed straight to Roboflow. It is not printed or written to any file.

In [ ]:
# 3. Download the newest version of each set in YOLO box format
import yaml
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow
rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))

downloads = {}   # short name -> (folder, {class number in that set: class name})
for short, (ws, proj_name) in SETS.items():
    proj = rf.workspace(ws).project(proj_name)
    ver = max(int(str(v.version).split("/")[-1]) for v in proj.versions())
    loc = proj.version(ver).download("yolov11", location=str(WORK / "raw" / short), overwrite=True).location
    names = yaml.safe_load(open(Path(loc) / "data.yaml"))["names"]
    names = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
    downloads[short] = (Path(loc), names)
    print(f"{short}: {ws}/{proj_name} version {ver}, their classes: {names}")

In [ ]:
# 4. List every photo with its boxes (their class names for now)
def read_boxes(label_file, names):
    boxes = []
    if not label_file.exists():
        return boxes
    for line in label_file.read_text().split("\n"):
        p = line.split()
        if len(p) < 5:
            continue
        cls = names.get(int(float(p[0])), "?")
        if len(p) == 5:
            x, y, w, h = map(float, p[1:])
        else:   # outline (polygon) labels: use the box around the outline
            xs, ys = [float(v) for v in p[1::2]], [float(v) for v in p[2::2]]
            x, y, w, h = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)
        boxes.append((cls, x, y, w, h))
    return boxes

raw = []
for short, (root, names) in downloads.items():
    for img in sorted(root.rglob("*")):
        if img.suffix.lower() in IMG_EXT and img.parent.name == "images":
            lab = img.parent.parent / "labels" / (img.stem + ".txt")
            raw.append({"path": img, "src": short, "boxes": read_boxes(lab, names)})
print("photo files found:", Counter(r["src"] for r in raw))

## Eye check of the labels
Twelve random photos per set with their boxes (their own class names). Look before trusting any score: boxes should sit on mould, not on healthy mycelium.

In [ ]:
# 5. Show 12 random boxed photos per set
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

def draw(rec):
    im = Image.open(rec["path"]).convert("RGB")
    d = ImageDraw.Draw(im)
    W, H = im.size
    for cls, x, y, w, h in rec["boxes"]:
        d.rectangle([(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H], outline=(255, 0, 0), width=max(2, W // 200))
    return im

for short in SETS:
    pool = [r for r in raw if r["src"] == short and r["boxes"]]
    pick = random.sample(pool, min(12, len(pool)))
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    for ax in axes.flat:
        ax.axis("off")
    for ax, r in zip(axes.flat, pick):
        ax.imshow(draw(r))
        ax.set_title(", ".join(sorted({b[0] for b in r["boxes"]})), fontsize=9)
    fig.suptitle(f"{short}: {len(pool)} photos with boxes", fontsize=14)
    plt.tight_layout()
    plt.show()

## Step 3. Remove copies and duplicates
First one copy per source photo (Roboflow names copies `<name>.rf.<code>.jpg`), then a perceptual hash across all three sets so resized or re-uploaded copies count too.

In [ ]:
# 6. One copy per source photo, then perceptual-hash duplicates across the sets
import numpy as np, imagehash

# a) Roboflow augmented copies: same text before ".rf."
seen, no_aug = set(), []
for r in raw:
    key = (r["src"], r["path"].name.split(".rf.")[0])
    if key not in seen:
        seen.add(key)
        no_aug.append(r)
print(f"files: {len(raw)} -> source photos after dropping Roboflow copies: {len(no_aug)}")

# b) perceptual hash
def phash_bits(path):
    with Image.open(path) as im:
        return imagehash.phash(im.convert("RGB")).hash.flatten().astype(np.int16)

recs, bits = [], []
for r in no_aug:
    try:
        bits.append(phash_bits(r["path"]))
        recs.append(r)
    except Exception as e:
        print("skipped unreadable photo:", r["path"].name, e)
B = np.stack(bits)
dist = 64 - (B @ B.T + (1 - B) @ (1 - B).T)

parent = list(range(len(recs)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
for i, j in np.argwhere(np.triu(dist <= HASH_DISTANCE, 1)):
    parent[find(i)] = find(j)

clusters = {}
for i in range(len(recs)):
    clusters.setdefault(find(i), []).append(i)
unique, cross_set = [], 0
for members in clusters.values():
    best = max(members, key=lambda i: (len(recs[i]["boxes"]), -i))   # keep the copy with most boxes
    unique.append(recs[best])
    cross_set += len({recs[i]["src"] for i in members}) > 1
print(f"duplicate groups spanning more than one set: {cross_set}")
print(f"UNIQUE PHOTOS: {len(unique)}  (from {len(raw)} files)")
print("unique photos per set:", dict(Counter(r["src"] for r in unique)))

## Step 4. Map the classes
Trichoderma stays **Trichoderma** (class 0). Mold, Cobweb and Neurospora become **mould** (class 1). Any other class name is dropped and listed, so nothing disappears silently. Photos left with no box are removed (they are not verified clean photos).

In [ ]:
# 7. Map classes and count boxes
import pandas as pd
mould_recs, dropped, rows = [], Counter(), []
no_box = 0
for r in unique:
    boxes = []
    for cls, x, y, w, h in r["boxes"]:
        k = CLASS_MAP.get(cls.strip().lower())
        rows.append((r["src"], cls, OUR_NAMES.get(k, "dropped")))
        if k is None:
            dropped[cls] += 1
        else:
            boxes.append((k, x, y, w, h))
    if boxes:
        mould_recs.append({"path": r["path"], "src": r["src"], "boxes": boxes})
    else:
        no_box += 1
df = pd.DataFrame(rows, columns=["set", "their class", "our class"])
print("boxes per their class:")
display(df.groupby(["their class", "our class"]).size().rename("boxes").reset_index())
print("boxes per our class:")
display(df.groupby("our class").size().rename("boxes"))
print("boxes per set and our class:")
display(pd.crosstab(df["set"], df["our class"]))
print("dropped class names (not mapped):", dict(dropped) or "none")
print(f"photos removed because no mapped box was left: {no_box}")
print(f"MOULD PHOTOS KEPT: {len(mould_recs)}")

## Step 5. Clean ("no mould") dish photos
The clean photos are **d005 only** (the private Kaggle dataset `d005-clean-dishes`). d003 is left out because its images are thin curve strips (about 78×435 px), not whole dishes.

**>>> CHECK THE TWO LINES BELOW <<<** Add the dataset (right panel, *Add Input*) and check `CLEAN_DIR` with the copy icon in the Input panel.

`CLEAN_ID_REGEX` picks the **dish id** out of each file path (one dish = one substrate batch), so that all photos of one dish go to the same side of the split. A few files (for example `FD_0009_Layer 1.jpg`) have no date in the name; they join the dish of the other photos in their folder. The notebook prints photos per dish: check that table. If a dish id cannot be read, it stops instead of guessing.

In [ ]:
# 8. >>> CHECK HERE <<<  clean dish photos (images with no mould = no boxes)
# Check the path with the copy icon in Kaggle's Input panel (next to the dataset name).
CLEAN_DIR = "/kaggle/input/d005-clean-dishes"
# One dish = one substrate batch. Example: FD_0000_180215_FD_Tv_180216.png.jpg -> "180215_FD_Tv"
CLEAN_ID_REGEX = r"(\d{6}_[A-Za-z]+_Tv)"

files = sorted(p for p in Path(CLEAN_DIR).rglob("*") if p.suffix.lower() in IMG_EXT)
assert files, f"no photos found under {CLEAN_DIR}: check that path (copy icon in Kaggle's Input panel)"
bad = [str(p) for p in files if "syldd" in str(p).lower()]
assert not bad, f"syldd photos must not be used here, e.g. {bad[0]}"
by_dish, loose = {}, []
for p in files:
    m = re.search(CLEAN_ID_REGEX, str(p.relative_to(CLEAN_DIR)))
    if m:
        by_dish.setdefault(m.group(1), []).append(p)
    else:
        loose.append(p)
# files without a date in the name (e.g. "FD_0009_Layer 1.jpg") join the one dish found in their folder
for p in loose:
    ids = {k for k, v in by_dish.items() if any(q.parent == p.parent for q in v)}
    if len(ids) != 1:
        raise ValueError(f"cannot read a dish id from {p.name}; fix CLEAN_ID_REGEX (example path: {p.relative_to(CLEAN_DIR)})")
    by_dish[ids.pop()].append(p)
print(f"{len(loose)} file(s) without a date in the name were placed by folder")
print("photos per dish:")
print({k: len(v) for k, v in sorted(by_dish.items())})
print("Note: only about 2 dishes are held out, so the false-alarm count in step 7 is rough.")
assert len(by_dish) >= 5, "fewer than 5 dishes found: the dish id is probably wrong, check CLEAN_ID_REGEX"
dishes = sorted(by_dish)
random.Random(SEED).shuffle(dishes)
n_val = max(1, round(HELD_OUT * len(dishes)))
clean_train, clean_val = [], []
for k, d in enumerate(dishes):
    side = clean_val if k < n_val else clean_train
    side += [{"path": p, "src": "clean", "boxes": []} for p in by_dish[d]]
print(f"clean photos: {len(clean_train)} for training ({len(dishes) - n_val} dishes), {len(clean_val)} held out ({n_val} dishes)")

## Step 6. Split, train, score
Mould photos: random 80/20 by photo (duplicates are already gone). Clean photos: split by dish (step 5). The reported precision, recall and mAP50 are on the held-out **mould** photos only; held-out clean photos are used in step 7.

In [ ]:
# 9. Helpers: build a YOLO dataset folder and train / score
def write_split(out, split, items):
    (out / "images" / split).mkdir(parents=True, exist_ok=True)
    (out / "labels" / split).mkdir(parents=True, exist_ok=True)
    for i, r in enumerate(items):
        stem = f"{r['src']}_{i}"
        shutil.copy(r["path"], out / "images" / split / (stem + r["path"].suffix.lower()))
        (out / "labels" / split / (stem + ".txt")).write_text("".join(f"{c} {x} {y} {w} {h}\n" for c, x, y, w, h in r["boxes"]))

def make_dataset(out, train_items, val_items):
    shutil.rmtree(out, ignore_errors=True)
    write_split(out, "train", train_items)
    write_split(out, "val", val_items)
    (out / "data.yaml").write_text(f"path: {out}\ntrain: images/train\nval: images/val\nnames:\n  0: Trichoderma\n  1: mould\n")
    return out / "data.yaml"

def train_and_score(train_items, val_items, tag, epochs=EPOCHS):
    from ultralytics import YOLO
    data = make_dataset(WORK / f"ds_{tag}", train_items, val_items)
    YOLO(MODEL).train(data=str(data), epochs=epochs, imgsz=IMG_SIZE, seed=SEED, project=str(WORK / "runs"),
                      name=tag, exist_ok=True, verbose=False, plots=False)
    best = WORK / "runs" / tag / "weights" / "best.pt"
    model = YOLO(str(best))
    b = model.val(data=str(data), imgsz=IMG_SIZE, verbose=False, plots=False).box
    per_class = {OUR_NAMES[int(c)]: round(float(a), 3) for c, a in zip(b.ap_class_index, b.ap50)}
    scores = {"precision": round(float(b.mp), 3), "recall": round(float(b.mr), 3),
              "mAP50": round(float(b.map50), 3), "mAP50 per class": per_class}
    return model, best, scores

# 10. 80/20 split by photo, then train
shuffled = mould_recs[:]
random.Random(SEED).shuffle(shuffled)
n_val = max(1, round(HELD_OUT * len(shuffled)))
val_mould, train_mould = shuffled[:n_val], shuffled[n_val:]
print(f"mould photos: {len(train_mould)} train, {len(val_mould)} held out")
print("held-out boxes per class:", dict(Counter(OUR_NAMES[b[0]] for r in val_mould for b in r["boxes"])))
model, best_path, scores = train_and_score(train_mould + clean_train, val_mould, "mould")

In [ ]:
# 11. Score table (held-out mould photos)
print(json.dumps(scores, indent=2))

## Step 7. False-alarm check
How many held-out **clean** dish photos get any mould box? Lower is better. A confidence of 0.25 is the loose setting, 0.5 the strict one.

In [ ]:
# 12. False alarms on held-out clean photos
false_alarms = {}
if not clean_val:
    print("No held-out clean photos (CLEAN_DIR placeholder not set): false-alarm check skipped.")
else:
    paths = [str(r["path"]) for r in clean_val]
    for conf in (0.25, 0.5):
        hits = 0
        for res in model.predict(paths, conf=conf, imgsz=IMG_SIZE, verbose=False, stream=True):
            hits += len(res.boxes) > 0
        false_alarms[str(conf)] = {"photos with a mould box": hits, "clean photos": len(paths)}
        print(f"confidence {conf}: {hits} of {len(paths)} clean photos get a mould box ({hits / len(paths):.1%})")

## Step 8. Save the weights
`mould_best.pt` and `mould_results.json` appear in the Output tab. Photo folders are deleted so the output stays small.

In [ ]:
# 13. Save weights and results, clean up photo folders
shutil.copy(best_path, WORK / "mould_best.pt")
json.dump({"scores_held_out_mould": scores, "false_alarms_clean": false_alarms,
           "mould_photos": {"train": len(train_mould), "held_out": len(val_mould)},
           "clean_photos": {"train": len(clean_train), "held_out": len(clean_val)},
           "settings": {"model": MODEL, "epochs": EPOCHS, "imgsz": IMG_SIZE, "hash_distance": HASH_DISTANCE, "seed": SEED}},
          open(WORK / "mould_results.json", "w"), indent=2)
for d in ["raw", "ds_mould", "runs"]:
    shutil.rmtree(WORK / d, ignore_errors=True)
print("saved:", sorted(p.name for p in WORK.iterdir()))

## Step 9. Leave one source out (switched off for now)
Today all three sets come from one owner, so leaving one out would test little. Once a **second licensed mould set** is added to `SETS` (and `mould_recs` holds its photos under its own `src` name), remove the `#` marks below. Each source in turn is held out entirely and the model is trained on the others plus the clean training photos. This is the test that shows whether the model works on mould photos from a different place.

In [ ]:
# 14. Leave-one-source-out (uncomment when a second licensed mould set is in)
# loso = []
# for held in sorted({r["src"] for r in mould_recs}):
#     tr = [r for r in mould_recs if r["src"] != held] + clean_train
#     te = [r for r in mould_recs if r["src"] == held]
#     _, _, s = train_and_score(tr, te, f"loso_{held}")
#     loso.append({"left-out source": held, "photos": len(te), **s})
# display(pd.DataFrame(loso))